# Epic 0 Evaluation Notebook

This notebook is the qualitative and reporting surface for the bbox-level CBIR MVP. It does not compute retrieval metrics ad hoc. Retrieval evaluation is generated by `mvp/evaluate.py`, and this notebook loads the resulting CSV/JSON/PNG artifacts for inspection and comparison.

Canonical experiment split:

- `exp01`: `Traineira` train gallery -> `Traineira` test queries. This is intra-class similarity calibration, not classification.
- `exp02`: `Traineira + Lancha / Iate` train gallery -> test queries. This is the first discriminative benchmark.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

cwd = Path.cwd().resolve()
REPO_ROOT = cwd.parent if cwd.name == "mvp" else cwd
MVP_ROOT = REPO_ROOT / "mvp"
if str(MVP_ROOT) not in sys.path:
    sys.path.insert(0, str(MVP_ROOT))

from data_prep import load_manifest, select_manifest_records, slugify_label
from visualize import render_inspect

target_class = "Traineira"
comparison_class = "Lancha / Iate"
split = "test"
idx = 1
padding_ratio = 0.0
benchmark_only = True

experiment_id = "exp02"
evaluation_dir = REPO_ROOT / "artifacts" / "evaluation" / experiment_id

manifest_paths = {
    target_class: REPO_ROOT / "data" / "cbir" / slugify_label(target_class) / "v1" / "manifest.jsonl",
    comparison_class: REPO_ROOT / "data" / "cbir" / slugify_label(comparison_class) / "v1" / "manifest.jsonl",
}

display(Markdown(f"**Repository:** `{REPO_ROOT}`"))
display(Markdown(f"**Evaluation artifact directory:** `{evaluation_dir}`"))


## 1. Manifest Audit

Use this block to confirm which class manifests exist and how many examples are available by split and benchmark flag.


In [ ]:
manifest_frames = []
for class_name, path in manifest_paths.items():
    if not path.exists():
        display(Markdown(f"Missing manifest for **{class_name}**: `{path}`"))
        continue
    records = load_manifest(path)
    df = pd.DataFrame(records)
    df["manifest_class"] = class_name
    manifest_frames.append(df)

if manifest_frames:
    manifests_df = pd.concat(manifest_frames, ignore_index=True)
    display(
        manifests_df.groupby(["target_class", "split", "is_benchmark_candidate"])
        .size()
        .rename("records")
        .reset_index()
    )
    display(
        manifests_df.groupby(["target_class", "camera_id", "size_bucket"])
        .size()
        .rename("records")
        .reset_index()
        .sort_values(["target_class", "records"], ascending=[True, False])
        .head(30)
    )
else:
    manifests_df = pd.DataFrame()


## 2. Load Evaluation Outputs

Run `mvp/evaluate.py` first. This notebook expects the generated artifacts below.


In [ ]:
summary_path = evaluation_dir / "summary.json"
ranking_path = evaluation_dir / "ranking.csv"
query_summary_path = evaluation_dir / "query_summary.csv"
threshold_distribution_path = evaluation_dir / "threshold_distribution.csv"
top1_confusion_path = evaluation_dir / "top1_confusion.csv"

if not summary_path.exists():
    display(Markdown(
        "Evaluation outputs were not found. Generate them with `uv run python mvp/evaluate.py ...` before using this section."
    ))
    summary = {}
else:
    summary = json.loads(summary_path.read_text())
    display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))

ranking_df = pd.read_csv(ranking_path) if ranking_path.exists() else pd.DataFrame()
query_summary_df = pd.read_csv(query_summary_path) if query_summary_path.exists() else pd.DataFrame()
threshold_distribution_df = pd.read_csv(threshold_distribution_path) if threshold_distribution_path.exists() else pd.DataFrame()
top1_confusion_df = pd.read_csv(top1_confusion_path, index_col=0) if top1_confusion_path.exists() else pd.DataFrame()


## 3. Ranking Tables

The main table has one row per retrieved hit. Threshold columns are booleans, so they can be aggregated directly.


In [ ]:
if ranking_df.empty:
    display(Markdown("No `ranking.csv` loaded."))
else:
    display(ranking_df.head(20))
    display(ranking_df.dtypes.rename("dtype").reset_index().rename(columns={"index": "column"}))

if not query_summary_df.empty:
    display(query_summary_df.head(20))

if not threshold_distribution_df.empty:
    display(threshold_distribution_df.head(30))

if not top1_confusion_df.empty:
    display(top1_confusion_df)


## 4. Evaluation Plots

These plots are generated by `evaluate.py`; the notebook only renders them.


In [ ]:
plot_paths = [
    evaluation_dir / "score_histogram.png",
    evaluation_dir / "top1_score_distribution.png",
    evaluation_dir / "threshold_rate_by_rank.png",
]
for plot_path in plot_paths:
    if plot_path.exists():
        display(Markdown(f"### {plot_path.name}"))
        display(Image(filename=str(plot_path)))


## 5. Inspect One Query Item

This remains a qualitative bbox audit step. It uses the manifest and runtime crop code, not the evaluation metrics code.


In [ ]:
manifest_path = manifest_paths[target_class]
if manifest_path.exists():
    class_records = load_manifest(manifest_path)
    selected_records = select_manifest_records(class_records, split=split, benchmark_only=benchmark_only)
    if selected_records:
        if idx < 1 or idx > len(selected_records):
            raise IndexError(f"idx must be between 1 and {len(selected_records)}; got {idx}")
        record = selected_records[idx - 1]
        figure, report = render_inspect(record=record, padding_ratio=padding_ratio)
        display(figure)
        plt.close(figure)
        display(pd.DataFrame({
            "field": [line.split(":", 1)[0] for line in report],
            "value": [line.split(":", 1)[1].strip() for line in report],
        }))
    else:
        display(Markdown("No records match the selected item filters."))
else:
    display(Markdown(f"Manifest not found: `{manifest_path}`"))


## 6. Compare Official Experiments

This block compares summaries if both experiment artifact folders exist.


In [ ]:
experiment_ids = ["exp01", "exp02", "exp02_smoke"]
summary_rows = []
for candidate_id in experiment_ids:
    candidate_path = REPO_ROOT / "artifacts" / "evaluation" / candidate_id / "summary.json"
    if not candidate_path.exists():
        continue
    payload = json.loads(candidate_path.read_text())
    metrics = payload.get("metrics", {})
    summary_rows.append({
        "experiment": candidate_id,
        "query_count": payload.get("query_count"),
        "ranking_rows": payload.get("ranking_rows"),
        "classes_in_queries": ", ".join(payload.get("classes_in_queries", [])),
        "classes_in_hits": ", ".join(payload.get("classes_in_hits", [])),
        "single_class_calibration": payload.get("is_single_class_calibration"),
        "top1_accuracy": metrics.get("top1_accuracy"),
        "precision_at_5": metrics.get("precision_at_5"),
        "precision_at_10": metrics.get("precision_at_10"),
        "precision_at_30": metrics.get("precision_at_30"),
        "mrr": metrics.get("mrr"),
        "remaining_self_hits": payload.get("leakage_checks", {}).get("remaining_self_hits"),
        "same_image_filename_hits": payload.get("leakage_checks", {}).get("same_image_filename_hits"),
    })

if summary_rows:
    display(pd.DataFrame(summary_rows))
else:
    display(Markdown("No experiment summaries found yet."))
